# 02 — Feature Engineering

Extends `02_feature_engineering.ipynb` with:

1. **Angle & counter sensors** — Uses each farm's `*-feature-description` Delta table (`is_angle`, `is_counter`). Angles use **circular** rolling means (via sin/cos). Counters use **per-step increments** (with reset handling) before rolling stats.
2. **Variance filtering (CELL 3)** — Drops near–zero variance sensors using **normal-status `train` rows only** (variance floor), then keeps Farm C at most **`MAX_SENSORS_AFTER_VAR`** by variance.
3. **Row scope (CELL 4)** — **All** telemetry rows (all status codes) are kept so status history, rolling windows, and labels can represent real transitions. **Isolation Forest** still trains on **`if_fit_eligible`** only; **CELL 7** deviation baselines and **CELL 9** z-score fits use **normal** `train` rows only.
4. **Risk labels** — Joins `*-event-info` to add `hours_to_next_anomaly` and graduated **`risk_score`** ∈ {0.0, 0.3, 0.6, 0.9, 1.0} from time until the next **anomaly** `event_start` (see `RISK_TIER_HOURS`).
5. **Rolling windows (`ROLLING_PARTITION_BY_SPLIT` in CELL 1)** — `True`: windows use `partitionBy(asset_id, train_test)` so rows never cross splits (training-safe; can create **discontinuities** when splits overlap in calendar time). `False` (**default**): windows use `partitionBy(asset_id)` for **operational continuity** (preferred when `train_test` overlaps in time); **leakage** into future split rows is possible — mitigate by evaluating on `if_fit_eligible` and time ordering. Details: **`NOTEBOOKS.md`**.
6. **Shared training mask** — After risk labels, each output table includes **`in_anomaly_window`** and **`if_fit_eligible`** (see `TRAIN_END` in CELL 1). Use the same columns for IF, LSTM, and XGBoost so all models agree on “clean” training rows.
7. **Status history (CELL 5)** — Downtime/derate **fractions** (24h / 7d / 30d), **status-change counts** (7d / 30d), **hours since last downtime** — interpretable units; not z-scaled in CELL 9.
8. **Dual horizons + operator target (CELL 8)** — **`hours_to_next_anomaly`** (next logged anomaly window) and **`hours_to_anomaly_linked_downtime`** (first `STATUS_DOWNTIME` inside that window). **`hours_to_next_operator_warning`** = sooner of the two (with null handling); tiered risk scores mirror `RISK_TIER_HOURS`. Interpretation caveats: see **`NOTEBOOKS.md`**.

**Outputs** (separate tables): `wind-farm-{a,b,c}-features`

**Validation & limitations** — LOOCV / small-N supervised, `TRAIN_END`, rolling tradeoffs, IF vs onset labels: see **`NOTEBOOKS.md`** in this folder.

> **Genie / docs:** Aligns with competition event tables (`event_label`, `event_start`, …) and feature metadata in silver.


In [ ]:
# CELL 1 — Imports and config

from pyspark.sql import functions as F, Window
from pyspark.sql.types import DoubleType
import pandas as pd
import warnings
import pickle
import os
warnings.filterwarnings("ignore")

CATALOG = "wind-turbine-silver"

FARM_CONFIG = {
    "a": {
        "table": "wind-farm-a",
        "output": "wind-farm-a-features",
        "feature_desc": "wind-farm-a-feature-description",
        "event_info": "wind-farm-a-event-info",
        "normal_status": [0],
    },
    "b": {
        "table": "wind-farm-b",
        "output": "wind-farm-b-features",
        "feature_desc": "wind-farm-b-feature-description",
        "event_info": "wind-farm-b-event-info",
        "normal_status": None,
    },
    "c": {
        "table": "wind-farm-c",
        "output": "wind-farm-c-features",
        "feature_desc": "wind-farm-c-feature-description",
        "event_info": "wind-farm-c-event-info",
        "normal_status": None,
    },
}

EXCLUDE_COLS = ["time_stamp", "asset_id", "id", "train_test", "status_type_id"]

# --- Feature pipeline knobs ---
# Drop sensors whose train-set variance is below this (after casting to double).
MIN_SENSOR_VARIANCE = 1e-10
# After variance floor, keep at most this many sensors on Farm C (by descending variance).
MAX_SENSORS_AFTER_VAR = 100

# Risk tiers: hours_to_next_anomaly -> score (upper bounds exclusive except last)
# > 336h -> 0.0 (handled as "no score" or beyond horizon); see helper below.
RISK_TIER_HOURS = [
    (24, 1.0),
    (72, 0.9),
    (168, 0.6),
    (336, 0.3),
]

# Rolling windows: True = partition by train_test (no cross-split leakage; discontinuities at split boundaries).
# False = continuity across calendar time (better when train_test overlaps in time; leakage risk — evaluate on if_fit_eligible).
# Default False per EDA review; see NOTEBOOKS.md.
ROLLING_PARTITION_BY_SPLIT = False

# Temporal cutoff for if_fit_eligible (shared with 03a_isolation_forest CELL 1 — keep identical).
# Set after profiling train_test / event timestamps (see 03a CELL 2a or silver queries).
TRAIN_END = "2023-01-01"

# Supervised (03b XGBoost): Farm A — all 12 anomalies with predictable anomaly-linked downtime transitions (EDA)
USABLE_ANOMALY_EVENT_IDS = {
    "a": frozenset({0, 10, 22, 26, 40, 42, 45, 51, 68, 72, 73, 84}),
    "b": frozenset(),
    "c": frozenset(),
}
EXCLUDE_OVERLAPPING_FROM_SUPERVISED_LABELS = True

# Status codes for history features and anomaly-linked downtime (edit if silver schema differs)
STATUS_DOWNTIME = [4]
STATUS_DERATED = [1]


In [ ]:
# CELL 2 — Schema validation

dfs_raw = {}

for farm, cfg in FARM_CONFIG.items():
    table_name = cfg["table"]
    df = spark.table(f"`{CATALOG}`.`{table_name}`")
    dfs_raw[farm] = df

    row_count = df.count()
    print(f"\n{'=' * 60}")
    print(f"Farm {farm.upper()} — `{CATALOG}`.`{table_name}`")
    print(f"{'=' * 60}")
    print(f"Row count: {row_count:,}")
    print(f"All columns ({len(df.columns)}): {df.columns}\n")

    required = ["time_stamp", "asset_id", "id", "train_test", "status_type_id"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(
            f"Farm {farm} missing required columns: {missing}\n"
            f"Available columns: {df.columns}"
        )
    print("All required columns present")

    print("\nNull counts for key columns:")
    df.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in required]).show()

    print(f"Farm {farm.upper()} status_type_id counts:")
    df.groupBy("status_type_id").count().orderBy("status_type_id").show()

    print(f"Farm {farm.upper()} train/test split:")
    df.groupBy("train_test").count().show()

    train_count = df.filter(F.col("train_test") == "train").count()
    if train_count == 0:
        raise ValueError(f"Farm {farm} has zero train rows — cannot proceed")
    print(f"Farm {farm.upper()} has {train_count:,} train rows")

print("\n>>> Review status_type distributions above.")
print(">>> Set FARM_CONFIG['b']['normal_status'] and FARM_CONFIG['c']['normal_status'] in Cell 3.")


In [ ]:
# CELL 3 — Feature descriptions, normal_status, sensor list + variance filter

# --- manual status (from Cell 2) ---
FARM_CONFIG["b"]["normal_status"] = [0]
FARM_CONFIG["c"]["normal_status"] = [0]


def base_sensor_name(col_name: str) -> str:
    # Map e.g. sensor_12_avg -> sensor_12, power_2_avg -> power_2.
    if col_name.endswith("_avg"):
        return col_name[: -len("_avg")]
    return col_name


def load_description_maps(farm: str):
    # Returns dict: base_sensor_name -> is_angle, is_counter (booleans).
    tbl = FARM_CONFIG[farm]["feature_desc"]
    desc = spark.table(f"`{CATALOG}`.`{tbl}`")
    pdf = desc.toPandas()
    # normalize boolean columns
    out = {}
    for _, row in pdf.iterrows():
        sn = str(row.get("sensor_name", "")).strip()
        if not sn:
            continue
        ia = bool(row.get("is_angle", False))
        ic = bool(row.get("is_counter", False))
        # handle string "true"/"false" from CSV
        if isinstance(row.get("is_angle"), str):
            ia = row.get("is_angle", "").lower() == "true"
        if isinstance(row.get("is_counter"), str):
            ic = row.get("is_counter", "").lower() == "true"
        out[sn] = {"is_angle": ia, "is_counter": ic}
    return out


desc_maps = {f: load_description_maps(f) for f in ["a", "b", "c"]}

sensor_cols = {}

for farm in ["a", "b", "c"]:
    df = dfs_raw[farm]
    cols = sorted(
        [
            c
            for c in df.columns
            if c not in EXCLUDE_COLS and not c.endswith(("_max", "_min", "_std"))
        ]
    )
    if len(cols) == 0:
        raise ValueError(f"No sensor columns found in Farm {farm.upper()}")

    cfg = FARM_CONFIG[farm]
    if cfg["normal_status"] is None:
        raise ValueError(f"Farm {farm}: normal_status is None — set FARM_CONFIG in Cell 3")
    train_df = df.filter(
        (F.col("train_test") == "train")
        & (F.col("status_type_id").isin(cfg["normal_status"]))
    )
    var_exprs = [F.variance(F.col(c).cast("double")).alias(c) for c in cols]
    variances = train_df.select(var_exprs).first().asDict()

    keep = []
    for c in cols:
        v = variances.get(c)
        v = 0.0 if v is None else float(v)
        if v >= MIN_SENSOR_VARIANCE:
            keep.append((c, v))
        else:
            print(f"  [{farm}] drop near-zero variance: {c} (var={v})")

    keep.sort(key=lambda x: -x[1])
    sensor_cols[farm] = [c for c, _ in keep]

    if farm == "c" and len(sensor_cols[farm]) > MAX_SENSORS_AFTER_VAR:
        sensor_cols[farm] = sensor_cols[farm][:MAX_SENSORS_AFTER_VAR]
        print(f"Farm C capped to {MAX_SENSORS_AFTER_VAR} sensors by variance ranking")

    print(
        f"Farm {farm.upper()}: {len(sensor_cols[farm])} sensors after variance filter "
        f"(min_var={MIN_SENSOR_VARIANCE})"
    )

# Column-level flags for rolling (keyed by actual DF column name)
sensor_flags = {}
for farm in ["a", "b", "c"]:
    sensor_flags[farm] = {}
    for c in sensor_cols[farm]:
        base = base_sensor_name(c)
        m = desc_maps[farm].get(base, {"is_angle": False, "is_counter": False})
        sensor_flags[farm][c] = m


In [ ]:
# CELL 4 — Keep all rows (all status codes on train and non-train splits)
# Downstream: Isolation Forest still trains only on if_fit_eligible (normal, pre-TRAIN_END, outside anomaly windows).
# Supervised labels and status-history features need downtime/derated/service rows in the table.

dfs = {}

for farm, cfg in FARM_CONFIG.items():
    df = dfs_raw[farm]
    if cfg["normal_status"] is None:
        raise ValueError(
            f"Farm {farm}: normal_status is None — update FARM_CONFIG in Cell 3 first"
        )

    n = df.count()
    print(f"Farm {farm.upper()}: {n:,} rows (all statuses retained)")
    dfs[farm] = df


In [ ]:
# CELL 5 — Preprocess angles/counters, then rolling features


def add_status_history_features(df):
    """Per-row operational history: downtime/derate fractions, status-change counts, hours since last downtime.
    Uses time-based range windows on unix time (assumes ~regular sampling; row fraction approximates time fraction).
    Not z-scaled in CELL 9."""
    out = df.withColumn("ts_u", F.unix_timestamp(F.col("time_stamp")))
    is_down = F.col("status_type_id").isin(STATUS_DOWNTIME).cast("double")
    is_der = (
        F.col("status_type_id").isin(STATUS_DERATED).cast("double")
        if STATUS_DERATED
        else F.lit(0.0)
    )
    w_order = Window.partitionBy("asset_id").orderBy(F.col("ts_u"))
    chg = F.when(
        F.col("status_type_id") != F.lag(F.col("status_type_id")).over(w_order),
        F.lit(1.0),
    ).otherwise(F.lit(0.0))
    out = out.withColumn("_chg", chg)

    def rw(sec):
        return Window.partitionBy("asset_id").orderBy(F.col("ts_u")).rangeBetween(-sec, 0)

    out = (
        out.withColumn("downtime_frac_24h", F.avg(is_down).over(rw(86400)))
        .withColumn("downtime_frac_7d", F.avg(is_down).over(rw(86400 * 7)))
        .withColumn("downtime_frac_30d", F.avg(is_down).over(rw(86400 * 30)))
        .withColumn("derated_frac_24h", F.avg(is_der).over(rw(86400)))
        .withColumn("derated_frac_7d", F.avg(is_der).over(rw(86400 * 7)))
        .withColumn("derated_frac_30d", F.avg(is_der).over(rw(86400 * 30)))
        .withColumn("status_change_count_7d", F.sum(F.col("_chg")).over(rw(86400 * 7)))
        .withColumn("status_change_count_30d", F.sum(F.col("_chg")).over(rw(86400 * 30)))
    )
    max_d_ts = F.max(
        F.when(F.col("status_type_id").isin(STATUS_DOWNTIME), F.col("ts_u"))
    ).over(w_order.rowsBetween(Window.unboundedPreceding, 0))
    out = out.withColumn(
        "hours_since_last_downtime",
        (F.col("ts_u") - max_d_ts) / F.lit(3600.0),
    ).drop("ts_u", "_chg")
    return out


def preprocess_angles_counters(df, cols, flags):
    # Add _work columns used for rolling (does not drop originals).
    w_order = Window.partitionBy("asset_id").orderBy("time_stamp")
    out = df
    work = {}  # col -> name of column to use for rolling
    for c in cols:
        fl = flags.get(c, {})
        if fl.get("is_counter"):
            lag_c = F.lag(F.col(c).cast("double"), 1).over(w_order)
            diff = F.col(c).cast("double") - lag_c
            rate = F.when(lag_c.isNull(), F.lit(0.0)).when(diff < 0, F.lit(0.0)).otherwise(diff)
            wname = f"{c}__counter_rate"
            out = out.withColumn(wname, rate)
            work[c] = wname
        elif fl.get("is_angle"):
            rad = F.radians(F.col(c).cast("double"))
            sname = f"{c}__sin"
            cname = f"{c}__cos"
            out = out.withColumn(sname, F.sin(rad)).withColumn(cname, F.cos(rad))
            work[c] = (sname, cname)  # tuple marker for circular
        else:
            work[c] = c
    return out, work


def add_rolling_for_subset(sub_df, cols, work, roll_partition_cols):
    # Compute rolling columns for one train_test subset (or full frame).
    if len(sub_df.take(1)) == 0:
        return sub_df
    order_window = Window.partitionBy(*roll_partition_cols).orderBy("time_stamp")
    w6 = order_window.rowsBetween(-5, 0)
    w36 = order_window.rowsBetween(-35, 0)
    w144 = order_window.rowsBetween(-143, 0)

    roll_exprs = []
    for c in cols:
        wspec = work[c]
        if isinstance(wspec, tuple):
            sname, cname = wspec
            sm6 = F.avg(sname).over(w6)
            cm6 = F.avg(cname).over(w6)
            sm36 = F.avg(sname).over(w36)
            cm36 = F.avg(cname).over(w36)
            sm144 = F.avg(sname).over(w144)
            cm144 = F.avg(cname).over(w144)
            circ_mean = lambda sm, cm: F.degrees(F.atan2(sm, cm))
            # Circular dispersion (degrees): sqrt(-2 ln R), R = mean resultant length
            R6 = F.sqrt(sm6 * sm6 + cm6 * cm6)
            circ_std_6 = F.when(R6 <= 1e-12, F.lit(0.0)).otherwise(
                F.degrees(F.sqrt(F.greatest(F.lit(0.0), -2 * F.log(F.greatest(F.lit(1e-12), R6)))))
            )
            ang = F.col(c).cast("double")
            lag_a = F.lag(ang, 1).over(order_window)
            d = ang - lag_a
            delta_unwrapped = (
                F.when(lag_a.isNull(), F.lit(None).cast("double"))
                .when(d > 180, d - 360)
                .when(d < -180, d + 360)
                .otherwise(d)
            )
            roll_exprs.extend(
                [
                    circ_mean(sm6, cm6).alias(f"{c}_roll_mean_1h"),
                    circ_mean(sm36, cm36).alias(f"{c}_roll_mean_6h"),
                    circ_mean(sm144, cm144).alias(f"{c}_roll_mean_24h"),
                    circ_std_6.alias(f"{c}_roll_std_6"),
                    delta_unwrapped.alias(f"{c}_delta"),
                ]
            )
        else:
            src = wspec
            roll_exprs.extend(
                [
                    F.avg(src).over(w6).alias(f"{c}_roll_mean_1h"),
                    F.avg(src).over(w36).alias(f"{c}_roll_mean_6h"),
                    F.avg(src).over(w144).alias(f"{c}_roll_mean_24h"),
                    F.stddev(src).over(w6).alias(f"{c}_roll_std_6"),
                    (F.col(src) - F.lag(F.col(src), 1).over(order_window)).alias(f"{c}_delta"),
                ]
            )

    out = sub_df.select("*", *roll_exprs)
    return out


for farm in ["a", "b", "c"]:
    df = dfs[farm]
    cols = sensor_cols[farm]
    flags = sensor_flags[farm]
    print(f"\nFarm {farm.upper()}: preprocessing + rolling for {len(cols)} sensors ...")

    cast_needed = [c for c in cols if df.schema[c].dataType != DoubleType()]
    if cast_needed:
        print(f"  Casting {len(cast_needed)} non-double columns to DoubleType")
        for c in cast_needed:
            df = df.withColumn(c, F.col(c).cast(DoubleType()))

    df, work = preprocess_angles_counters(df, cols, flags)

    from functools import reduce

    if ROLLING_PARTITION_BY_SPLIT:
        roll_part = ["asset_id", "train_test"]
        splits = df.select("train_test").distinct().rdd.map(lambda r: r[0]).collect()
        parts = []
        for split in splits:
            sub = df.filter(F.col("train_test") == split)
            parts.append(add_rolling_for_subset(sub, cols, work, roll_part))
        df = reduce(lambda a, b: a.unionByName(b), parts)
    else:
        roll_part = ["asset_id"]
        df = add_rolling_for_subset(df, cols, work, roll_part)

    new_roll_cols = [
        cn
        for cn in df.columns
        if cn.endswith(("_roll_mean_1h", "_roll_mean_6h", "_roll_mean_24h", "_roll_std_6", "_delta"))
    ]

    all_null_cond = F.lit(True)
    for rc in new_roll_cols:
        all_null_cond = all_null_cond & F.col(rc).isNull()

    before_ct = df.count()
    df = df.filter(~all_null_cond)
    after_ct = df.count()
    print(f"  Dropped {before_ct - after_ct:,} rows where ALL rolling cols are null")

    df = df.fillna(0, subset=new_roll_cols)

    # Drop helper cols for angles (sin/cos) to keep schema lean
    drop_helpers = [c for c in df.columns if c.endswith("__sin") or c.endswith("__cos") or c.endswith("__counter_rate")]
    if drop_helpers:
        df = df.drop(*drop_helpers)

    df = add_status_history_features(df)
    print("  Added status history features")

    dfs[farm] = df
    print(f"  Final: {after_ct:,} rows, {len(df.columns)} columns")


In [ ]:
# CELL 6 — Interaction terms (Farm A: gearbox oil × power/wind — no vibration sensors in metadata)

for farm in ["a", "b", "c"]:
    tbl = FARM_CONFIG[farm]["feature_desc"]
    desc_df = spark.table(f"`{CATALOG}`.`{tbl}`")
    desc_pd = desc_df.toPandas()

    a_columns = set(dfs[farm].columns)

    def _resolve_col(name):
        if name in a_columns:
            return name
        avg = f"{name}_avg"
        if avg in a_columns:
            return avg
        return None

    if farm == "a":
        m_gb = desc_pd["description"].str.contains("gearbox", case=False, na=False) & desc_pd[
            "description"
        ].str.contains("oil", case=False, na=False)
        gb_cols = [
            c
            for c in (_resolve_col(s) for s in desc_pd.loc[m_gb, "sensor_name"])
            if c is not None
        ]
        gb = gb_cols[0] if gb_cols else _resolve_col("sensor_12")
        m_pw = desc_pd["description"].str.contains("active power", case=False, na=False) & ~desc_pd[
            "description"
        ].str.contains("reactive", case=False, na=False)
        pw_cols = [
            c
            for c in (_resolve_col(s) for s in desc_pd.loc[m_pw, "sensor_name"])
            if c is not None
        ]
        pw = pw_cols[0] if pw_cols else (_resolve_col("sensor_29") or _resolve_col("sensor_30"))
        m_w = desc_pd["description"].str.contains("wind speed", case=False, na=False)
        w_cols = [
            c
            for c in (_resolve_col(s) for s in desc_pd.loc[m_w, "sensor_name"])
            if c is not None
        ]
        wnd = w_cols[0] if w_cols else (_resolve_col("sensor_3") or _resolve_col("sensor_4"))
        print(f"\nFarm A: gearbox_oil={gb}, power={pw}, wind={wnd}")
        dfa = dfs["a"]
        if gb and pw:
            dfa = dfa.withColumn("gearbox_oil_x_power", F.col(gb).cast("double") * F.col(pw).cast("double"))
        if gb and wnd:
            dfa = dfa.withColumn("gearbox_oil_x_wind", F.col(gb).cast("double") * F.col(wnd).cast("double"))
        dfs["a"] = dfa
        print("  Added Farm A gearbox load interactions (gearbox_oil_x_power / gearbox_oil_x_wind)")
        continue

    temp_mask = (
        desc_pd["description"].str.contains("temp", case=False, na=False)
        | desc_pd["unit"].str.contains("C", case=False, na=False)
    ) & ~desc_pd["description"].str.contains("ambient", case=False, na=False)
    temp_sensors = [
        c
        for c in (_resolve_col(s) for s in desc_pd.loc[temp_mask, "sensor_name"])
        if c is not None
    ]

    vib_mask = desc_pd["description"].str.contains("vibration", case=False, na=False) | desc_pd[
        "description"
    ].str.contains("rpm", case=False, na=False)
    vib_sensors = [
        c
        for c in (_resolve_col(s) for s in desc_pd.loc[vib_mask, "sensor_name"])
        if c is not None
    ]

    print(f"\nFarm {farm.upper()}: temp={temp_sensors[:3]}, vib={vib_sensors[:3]}")

    if temp_sensors and vib_sensors:
        second_vib = vib_sensors[1] if len(vib_sensors) > 1 else vib_sensors[0]
        dfs[farm] = (
            dfs[farm]
            .withColumn("oil_x_vib", F.col(temp_sensors[0]) * F.col(vib_sensors[0]))
            .withColumn("temp_x_rpm", F.col(temp_sensors[0]) * F.col(second_vib))
        )
        print(f"  Added oil_x_vib, temp_x_rpm")
    else:
        print("  Skipping interactions (missing matched columns)")


In [ ]:
# CELL 7 — Per-turbine baseline deviation (train-only means)

for farm in ["a", "b", "c"]:
    df = dfs[farm]
    cols = sensor_cols[farm]
    print(f"\nFarm {farm.upper()}: per-turbine deviation for {len(cols)} sensors ...")

    ns = FARM_CONFIG[farm]["normal_status"]
    turbine_means = df.filter(
        (F.col("train_test") == "train") & (F.col("status_type_id").isin(ns))
    ).groupBy("asset_id").agg(*[F.mean(F.col(c)).alias(f"{c}_mean") for c in cols])

    df = df.join(turbine_means, on="asset_id", how="left")

    keep_cols = [F.col(c) for c in df.columns if not c.endswith("_mean")]
    dev_exprs = [(F.col(c) - F.col(f"{c}_mean")).alias(f"{c}_deviation") for c in cols]
    df = df.select(*keep_cols, *dev_exprs)

    dfs[farm] = df
    print(f"  Added {len(cols)} deviation columns ({df.count():,} rows)")


In [ ]:
# CELL 8 — Risk labels from event_info (graduated risk_score)


def normalize_events(events_df):
    # Ensure asset_id column exists (Farm A may use `asset`).
    out = events_df
    if "asset_id" not in out.columns and "asset" in out.columns:
        out = out.withColumn("asset_id", F.col("asset"))
    return out


def add_risk_labels(df, event_table_name: str, farm: str):
    ev = normalize_events(spark.table(f"`{CATALOG}`.`{event_table_name}`"))
    ev_a = ev.filter(F.col("event_label") == "anomaly").select(
        F.col("event_id"),
        F.col("asset_id").cast("string").alias("ev_asset"),
        F.to_timestamp(F.col("event_start")).alias("event_start_ts"),
    )

    base = df.withColumn("_ts", F.to_timestamp(F.col("time_stamp")))
    # next anomaly strictly after this row's timestamp
    j = base.join(
        ev_a,
        (F.col("asset_id").cast("string") == F.col("ev_asset"))
        & (F.col("event_start_ts") > F.col("_ts")),
        how="left",
    )
    next_ts = j.groupBy([F.col("asset_id"), F.col("time_stamp"), F.col("id")]).agg(
        F.min(F.struct(F.col("event_start_ts"), F.col("event_id"))).alias("pick")
    )
    out = base.join(next_ts, on=["asset_id", "time_stamp", "id"], how="left")
    out = out.withColumn("next_anomaly_ts", F.col("pick.event_start_ts")).withColumn(
        "next_anomaly_event_id", F.col("pick.event_id")
    ).drop("pick")
    hours = (F.unix_timestamp("next_anomaly_ts") - F.unix_timestamp("_ts")) / F.lit(3600.0)
    out = out.withColumn("hours_to_next_anomaly", F.when(F.col("next_anomaly_ts").isNull(), None).otherwise(hours))

    h = F.col("hours_to_next_anomaly")
    out = out.withColumn(
        "risk_score",
        F.when(F.col("next_anomaly_ts").isNull(), F.lit(0.0))
        .when(h <= 24, F.lit(1.0))
        .when(h <= 72, F.lit(0.9))
        .when(h <= 168, F.lit(0.6))
        .when(h <= 336, F.lit(0.3))
        .otherwise(F.lit(0.0)),
    )
    usable_set = USABLE_ANOMALY_EVENT_IDS.get(farm, frozenset())
    if usable_set:
        ids = sorted(usable_set)
        out = out.withColumn(
            "is_usable_supervised_next",
            F.col("next_anomaly_event_id").isin(*ids),
        )
    else:
        out = out.withColumn("is_usable_supervised_next", F.lit(False))
    out = out.drop("_ts", "next_anomaly_ts")
    return out


def risk_tier_from_hours(h_col: str):
    h = F.col(h_col)
    return (
        F.when(h.isNull(), F.lit(0.0))
        .when(h <= 24, F.lit(1.0))
        .when(h <= 72, F.lit(0.9))
        .when(h <= 168, F.lit(0.6))
        .when(h <= 336, F.lit(0.3))
        .otherwise(F.lit(0.0))
    )


def compute_first_anomaly_downtime_ts(farm: str):
    """First timestamp where status is downtime (STATUS_DOWNTIME) inside each labeled anomaly id window."""
    cfg = FARM_CONFIG[farm]
    ev = normalize_events(spark.table(f"`{CATALOG}`.`{cfg['event_info']}`"))
    ev = ev.filter(F.col("event_label") == "anomaly").select(
        F.col("event_id"),
        F.col("asset_id").cast("string").alias("ev_asset"),
        F.col("event_start_id"),
        F.col("event_end_id"),
    )
    tel = dfs_raw[farm].select(
        F.col("asset_id").cast("string").alias("tel_asset"),
        F.col("id"),
        F.to_timestamp(F.col("time_stamp")).alias("tel_ts"),
        F.col("status_type_id"),
    )
    j = tel.join(
        ev,
        (F.col("tel_asset") == F.col("ev_asset"))
        & (F.col("id") >= F.col("event_start_id"))
        & (F.col("id") <= F.col("event_end_id")),
        "inner",
    )
    fd = j.filter(F.col("status_type_id").isin(STATUS_DOWNTIME)).groupBy(
        "event_id", "ev_asset", "event_start_id", "event_end_id"
    ).agg(F.min("tel_ts").alias("first_anomaly_downtime_ts"))
    return fd


def add_anomaly_linked_downtime_and_operator_targets(df, first_dt_df):
    ev_join = first_dt_df.select(
        F.col("event_id"),
        F.col("ev_asset"),
        F.col("first_anomaly_downtime_ts").alias("next_downtime_ts"),
    )
    base = df.withColumn("_ts", F.to_timestamp(F.col("time_stamp")))
    j = base.join(
        ev_join,
        (F.col("asset_id").cast("string") == F.col("ev_asset"))
        & (F.col("next_downtime_ts") > F.col("_ts")),
        how="left",
    )
    pick = j.groupBy("asset_id", "time_stamp", "id").agg(
        F.min(F.struct(F.col("next_downtime_ts"), F.col("event_id"))).alias("pick")
    )
    out = base.join(pick, on=["asset_id", "time_stamp", "id"], how="left")
    out = out.withColumn("next_anomaly_linked_downtime_ts", F.col("pick.next_downtime_ts")).drop("pick")
    hd = (
        F.unix_timestamp("next_anomaly_linked_downtime_ts") - F.unix_timestamp("_ts")
    ) / F.lit(3600.0)
    out = out.withColumn(
        "hours_to_anomaly_linked_downtime",
        F.when(F.col("next_anomaly_linked_downtime_ts").isNull(), None).otherwise(hd),
    )
    out = out.withColumn(
        "risk_score_anomaly_linked_downtime",
        risk_tier_from_hours("hours_to_anomaly_linked_downtime"),
    )
    ha = F.col("hours_to_next_anomaly")
    hdcol = F.col("hours_to_anomaly_linked_downtime")
    out = out.withColumn(
        "hours_to_next_operator_warning",
        F.when(ha.isNull() & hdcol.isNull(), F.lit(None).cast("double"))
        .when(ha.isNull(), hdcol)
        .when(hdcol.isNull(), ha)
        .otherwise(F.least(ha, hdcol)),
    )
    out = out.withColumn(
        "risk_score_operator_warning",
        risk_tier_from_hours("hours_to_next_operator_warning"),
    )
    return out.drop("_ts", "next_anomaly_linked_downtime_ts")


for farm in ["a", "b", "c"]:
    ename = FARM_CONFIG[farm]["event_info"]
    dfs[farm] = add_risk_labels(dfs[farm], ename, farm)
    print(f"Farm {farm.upper()}: joined risk labels from `{ename}`")
    dfs[farm].select("risk_score").describe().show()

for farm in ["a", "b", "c"]:
    fd = compute_first_anomaly_downtime_ts(farm)
    dfs[farm] = add_anomaly_linked_downtime_and_operator_targets(dfs[farm], fd)
    print(f"Farm {farm.upper()}: hours_to_anomaly_linked_downtime + hours_to_next_operator_warning")

# Anomaly id windows + fit eligibility (single definition for IF / LSTM / XGBoost downstream)
for farm in ["a", "b", "c"]:
    ename = FARM_CONFIG[farm]["event_info"]
    ev = normalize_events(spark.table(f"`{CATALOG}`.`{ename}`"))
    anom_ev = ev.filter(F.col("event_label") == "anomaly").select(
        "asset_id", "event_start_id", "event_end_id"
    )

    flagged = (
        dfs[farm].alias("d")
        .join(
            F.broadcast(anom_ev).alias("a"),
            (F.col("d.asset_id") == F.col("a.asset_id"))
            & (F.col("d.id") >= F.col("a.event_start_id"))
            & (F.col("d.id") <= F.col("a.event_end_id")),
            "inner",
        )
        .select(F.col("d.asset_id"), F.col("d.id"))
        .dropDuplicates()
    )

    dfs[farm] = (
        dfs[farm]
        .join(
            flagged.withColumn("in_anomaly_window", F.lit(True)),
            on=["asset_id", "id"],
            how="left",
        )
        .withColumn(
            "in_anomaly_window",
            F.coalesce(F.col("in_anomaly_window"), F.lit(False)),
        )
        .withColumn(
            "if_fit_eligible",
            (F.to_timestamp(F.col("time_stamp")) < F.to_timestamp(F.lit(TRAIN_END)))
            & (~F.col("in_anomaly_window"))
            & (F.col("status_type_id").isin(FARM_CONFIG[farm]["normal_status"])),
        )
    )
    print(f"Farm {farm.upper()}: added in_anomaly_window + if_fit_eligible")


In [ ]:
# CELL 9 — Standard-scaling (fit on train only; exclude label columns)

SCALER_DIR = "/tmp/DSMLC-Final-Comp-2026/scalers"
os.makedirs(SCALER_DIR, exist_ok=True)
print(f"Scaler stats -> {SCALER_DIR}")

# Not scaled (pass-through); booleans/masks are excluded from feature_cols by construction
LABEL_AND_ID_COLS = {
    "hours_to_next_anomaly",
    "risk_score",
    "hours_to_anomaly_linked_downtime",
    "risk_score_anomaly_linked_downtime",
    "hours_to_next_operator_warning",
    "risk_score_operator_warning",
    "next_anomaly_event_id",
    "is_usable_supervised_next",
    "downtime_frac_24h",
    "downtime_frac_7d",
    "downtime_frac_30d",
    "derated_frac_24h",
    "derated_frac_7d",
    "derated_frac_30d",
    "status_change_count_7d",
    "status_change_count_30d",
    "hours_since_last_downtime",
    "if_fit_eligible",
    "in_anomaly_window",
}

for farm in ["a", "b", "c"]:
    df = dfs[farm]
    cols = sensor_cols[farm]

    roll_cols = [
        c
        for c in df.columns
        if c.endswith(("_roll_mean_1h", "_roll_mean_6h", "_roll_mean_24h", "_roll_std_6", "_delta"))
    ]
    dev_cols = [c for c in df.columns if c.endswith("_deviation")]

    feature_cols = list(cols) + roll_cols + dev_cols
    for ic in [
        "oil_x_vib",
        "temp_x_rpm",
        "gearbox_oil_x_power",
        "gearbox_oil_x_wind",
    ]:
        if ic in df.columns:
            feature_cols.append(ic)

    feature_cols = sorted(set(feature_cols))

    print(f"\nFarm {farm.upper()}: scaling {len(feature_cols)} feature columns ...")

    df = df.fillna(0, subset=feature_cols)

    ns = FARM_CONFIG[farm]["normal_status"]
    train_df = df.filter(
        (F.col("train_test") == "train") & (F.col("status_type_id").isin(ns))
    )
    train_count = train_df.count()
    if train_count == 0:
        raise ValueError(f"Farm {farm}: no train rows — cannot fit scaler")
    print(f"  Fitting on {train_count:,} train rows")

    agg_exprs = []
    for c in feature_cols:
        agg_exprs.append(F.mean(c).alias(f"{c}__mean"))
        agg_exprs.append(F.stddev_pop(c).alias(f"{c}__std"))

    stats = train_df.select(agg_exprs).first().asDict()

    scaler_path = os.path.join(SCALER_DIR, f"scaler_{farm}.pkl")
    with open(scaler_path, "wb") as fh:
        pickle.dump(stats, fh)
    print(f"  Scaler stats saved to {scaler_path}")

    feat_set = set(feature_cols)
    col_exprs = []
    for c in df.columns:
        if c in feat_set:
            m = float(stats.get(f"{c}__mean") or 0.0)
            s = float(stats.get(f"{c}__std") or 1.0)
            if s == 0:
                s = 1.0
            col_exprs.append(((F.col(c) - F.lit(m)) / F.lit(s)).alias(c))
        else:
            col_exprs.append(F.col(c))

    df = df.select(*col_exprs)
    dfs[farm] = df
    print(f"  Scaling applied ({df.count():,} rows)")


In [ ]:
# CELL 10 — Output validation

REQUIRED_SUFFIXES = [
    "_roll_mean_1h",
    "_roll_mean_6h",
    "_roll_mean_24h",
    "_roll_std_6",
    "_delta",
    "_deviation",
]

for farm in ["a", "b", "c"]:
    df = dfs[farm]
    cols_set = set(df.columns)

    print(f"\n{'=' * 60}")
    print(f"Farm {farm.upper()} — output validation")
    print(f"{'=' * 60}")

    for rc in [
        "asset_id",
        "time_stamp",
        "id",
        "train_test",
        "status_type_id",
        "risk_score",
        "hours_to_next_anomaly",
        "hours_to_anomaly_linked_downtime",
        "risk_score_anomaly_linked_downtime",
        "hours_to_next_operator_warning",
        "risk_score_operator_warning",
        "downtime_frac_24h",
        "downtime_frac_7d",
        "downtime_frac_30d",
        "derated_frac_24h",
        "derated_frac_7d",
        "derated_frac_30d",
        "status_change_count_7d",
        "status_change_count_30d",
        "hours_since_last_downtime",
        "next_anomaly_event_id",
        "is_usable_supervised_next",
        "if_fit_eligible",
        "in_anomaly_window",
    ]:
        if rc not in cols_set:
            raise ValueError(f"Farm {farm} output missing column: {rc}")

    for sc in sensor_cols[farm]:
        if sc not in cols_set:
            raise ValueError(f"Farm {farm} output missing sensor column: {sc}")

    for sc in sensor_cols[farm]:
        for suffix in REQUIRED_SUFFIXES:
            expected = f"{sc}{suffix}"
            if expected not in cols_set:
                raise ValueError(f"Farm {farm} output missing column: {expected}")

    print(f"  Columns: {len(df.columns)}")
    row_ct = df.count()
    print(f"  Rows:    {row_ct:,}")
    print(f"  Farm {farm.upper()} PASSED validation")


In [ ]:
# CELL 11 — Save to Delta

for farm, cfg in FARM_CONFIG.items():
    out = cfg["output"]
    print(f"\nSaving Farm {farm.upper()} -> `{CATALOG}`.`{out}` ...")
    dfs[farm].write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(
        f"`{CATALOG}`.`{out}`"
    )
    cnt = spark.table(f"`{CATALOG}`.`{out}`").count()
    print(f"  saved {cnt:,} rows")
